In [1]:
# Install required packages and load environment variables
%pip install anthropic
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-6"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Helper functions â€” chat() now supports stop_sequences
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Tanpa teknik apapun â€” Claude jawab dengan markdown formatting
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
text

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["stopped"]\n  }\n}\n```\n\nThis rule triggers when an **EC2 instance is stopped**.'

In [4]:
# Teknik 1: System Prompt untuk kontrol format output
# Prefill tidak support di model Claude terbaru
# Alternatif: instruksikan lewat system prompt
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, system="Respond with only raw valid JSON. No markdown, no explanation, no code blocks.")
text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [5]:
# Teknik 2: Instruksi format di dalam user message
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json. Return only raw JSON, no markdown.")

text = chat(messages)
text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [6]:
# Hasilnya langsung bisa di-parse
import json

messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, system="Respond with only raw valid JSON. No markdown, no explanation, no code blocks.")
data = json.loads(text.strip())
print(json.dumps(data, indent=2))

{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "running"
    ]
  }
}
